# ZeraaTech Treatment Advisor Model

Train Random Forest to predict plant health status from sensor data and produce treatment advice.

**Dataset:** [Plant Health Data (ziya07) on Kaggle](https://www.kaggle.com/datasets/ziya07/plant-health-data)

**Features used (match ZeraaTech sensor kit):**
- Soil Moisture
- Ambient Temperature
- Humidity
- Soil pH

**Target:** `Plant_Health_Status` (Healthy / Moderate Stress / High Stress)

**Output for ZeraaTech:** classification + bilingual treatment text

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete.')

## 2. Load dataset

Download CSV from Kaggle and put in same folder as this notebook, OR use kaggle CLI:

```bash
pip install kaggle
kaggle datasets download -d ziya07/plant-health-data
unzip plant-health-data.zip
```

The file is named `plant_health_data.csv`.

In [ ]:
df = pd.read_csv('plant_health_data.csv')
print(f'Dataset shape: {df.shape}')
df.head()

In [ ]:
print('Columns:')
print(df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())

In [ ]:
print('Target class distribution:')
print(df['Plant_Health_Status'].value_counts())
print()
df['Plant_Health_Status'].value_counts().plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Plant Health Status Distribution')
plt.xlabel('Status')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Exploratory data analysis

In [ ]:
df.describe()

In [ ]:
# Feature distribution by class
features_to_plot = ['Soil_Moisture', 'Ambient_Temperature', 'Humidity', 'Soil_pH']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    for status in df['Plant_Health_Status'].unique():
        subset = df[df['Plant_Health_Status'] == status]
        axes[i].hist(subset[feature], bins=20, alpha=0.5, label=status)
    axes[i].set_title(f'{feature} by Plant Health Status')
    axes[i].set_xlabel(feature)
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

plt.tight_layout()
plt.savefig('feature_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numeric features only)
numeric_df = df.select_dtypes(include=[np.number])
plt.figure(figsize=(10, 8))
sns.heatmap(numeric_df.corr(), annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Feature selection

ZeraaTech sensor kit measures: soil moisture, temperature, humidity, pH.

Train ONLY on those four features to make sure the deployed model can run on what the actual ESP32 sends.

In [ ]:
FEATURES = ['Soil_Moisture', 'Ambient_Temperature', 'Humidity', 'Soil_pH']
TARGET = 'Plant_Health_Status'

X = df[FEATURES].copy()
y = df[TARGET].copy()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print()
print('Feature ranges:')
print(X.describe().loc[['min', 'max', 'mean']])
print()
print('Classes:')
print(y.unique())

## 5. Train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_SEED,
    stratify=y
)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Train class distribution:')
print(y_train.value_counts(normalize=True).round(3))
print(f'Test class distribution:')
print(y_test.value_counts(normalize=True).round(3))

## 6. Train and compare multiple classifiers

In [ ]:
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_SEED),
    'Decision Tree': DecisionTreeClassifier(random_state=RANDOM_SEED),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'SVM': SVC(random_state=RANDOM_SEED),
    'KNN': KNeighborsClassifier(n_neighbors=5),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'train_acc': train_acc,
        'test_acc': test_acc,
        'cv_mean': cv_scores.mean(),
        'cv_std': cv_scores.std()
    }
    print(f'{name:20s}  train={train_acc:.4f}  test={test_acc:.4f}  cv={cv_scores.mean():.4f}+/-{cv_scores.std():.4f}')

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# Bar chart compare models
fig, ax = plt.subplots(figsize=(10, 6))
results_df[['test_acc', 'cv_mean']].plot(kind='bar', ax=ax)
ax.set_title('Model Comparison: Treatment Advisor')
ax.set_ylabel('Accuracy')
ax.set_xlabel('Model')
ax.set_ylim(0, 1.0)
ax.legend(['Test Accuracy', 'CV Mean Accuracy'])
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Pick best model and evaluate

Random Forest usually win on this kind of tabular data. Pick it as final.

In [ ]:
final_model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=RANDOM_SEED)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
test_acc = accuracy_score(y_test, y_pred)
print(f'Final test accuracy: {test_acc:.4f}')
print()
print('Classification report:')
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred, labels=final_model.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=final_model.classes_)
fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_title(f'Confusion Matrix (Accuracy: {test_acc:.2%})')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance
importances = pd.Series(final_model.feature_importances_, index=FEATURES).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Feature Importance (Random Forest)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()
print(importances)

## 8. Save model + treatment advice mapping (bilingual)

In [ ]:
# Treatment advice in English and Arabic
treatment_advice = {
    'Healthy': {
        'en': 'Plant is healthy. Continue current irrigation and monitoring schedule.',
        'ar': 'النبات بصحة جيدة. استمر في جدول الري والمراقبة الحالي.'
    },
    'Moderate Stress': {
        'en': 'Plant is under moderate stress. Check soil moisture and consider light irrigation. Inspect for early signs of disease.',
        'ar': 'النبات يعاني من ضغط متوسط. تحقق من رطوبة التربة وفكر في الري الخفيف. افحص العلامات المبكرة للمرض.'
    },
    'High Stress': {
        'en': 'Plant is under high stress. Irrigate immediately, reduce sun exposure with shade cloth, and check pH balance. Consider soil treatment.',
        'ar': 'النبات في حالة ضغط حاد. اروي على الفور، قلل التعرض للشمس بقماش التظليل، وتحقق من توازن الأس الهيدروجيني. فكر في معالجة التربة.'
    }
}

# Save model
with open('treatment_model.pkl', 'wb') as f:
    pickle.dump(final_model, f)

# Save treatment advice
with open('treatment_advice.json', 'w', encoding='utf-8') as f:
    json.dump(treatment_advice, f, ensure_ascii=False, indent=2)

# Save feature names (for inference safety)
with open('treatment_features.json', 'w') as f:
    json.dump(FEATURES, f, indent=2)

print('Saved:')
print('  treatment_model.pkl')
print('  treatment_advice.json')
print('  treatment_features.json')

## 9. Test inference (simulate Flask endpoint call)

In [ ]:
# Load saved artifacts (verify they work)
with open('treatment_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)
with open('treatment_advice.json', 'r', encoding='utf-8') as f:
    loaded_advice = json.load(f)
with open('treatment_features.json', 'r') as f:
    loaded_features = json.load(f)

def predict_treatment(moisture, temperature, humidity, pH, lang='en'):
    """Predict plant health status and return treatment advice.
    
    Args:
        moisture: soil moisture, percent
        temperature: ambient temperature, Celsius
        humidity: relative humidity, percent
        pH: soil pH (use 6.5 default if no pH sensor)
        lang: 'en' or 'ar'
    """
    X_input = pd.DataFrame([[moisture, temperature, humidity, pH]], columns=loaded_features)
    status = loaded_model.predict(X_input)[0]
    confidence = loaded_model.predict_proba(X_input).max()
    advice = loaded_advice[status][lang]
    return {
        'status': status,
        'confidence': float(confidence),
        'advice': advice,
        'language': lang
    }

# Test case 1: healthy conditions
print('Test 1 (healthy):')
print(predict_treatment(moisture=60, temperature=24, humidity=65, pH=6.5, lang='en'))
print(predict_treatment(moisture=60, temperature=24, humidity=65, pH=6.5, lang='ar'))
print()

# Test case 2: high stress (dry, hot, acidic)
print('Test 2 (high stress):')
print(predict_treatment(moisture=15, temperature=42, humidity=20, pH=4.5, lang='en'))
print(predict_treatment(moisture=15, temperature=42, humidity=20, pH=4.5, lang='ar'))
print()

# Test case 3: moderate stress
print('Test 3 (moderate stress):')
print(predict_treatment(moisture=35, temperature=33, humidity=40, pH=6.0, lang='en'))

## 10. Flask endpoint code (copy into your AI service)

Add this to your existing Flask file alongside the disease and crop endpoints:

```python
import pickle, json
import pandas as pd
from flask import request, jsonify

# Load at startup
treatment_model = pickle.load(open('models/treatment_model.pkl', 'rb'))
treatment_advice = json.load(open('models/treatment_advice.json', encoding='utf-8'))
treatment_features = json.load(open('models/treatment_features.json'))

@app.route('/predict-treatment', methods=['POST'])
def predict_treatment_endpoint():
    data = request.json
    lang = data.get('lang', 'en')
    X_input = pd.DataFrame([[
        data['moisture'],
        data['temperature'],
        data['humidity'],
        data.get('pH', 6.5)
    ]], columns=treatment_features)
    status = treatment_model.predict(X_input)[0]
    confidence = float(treatment_model.predict_proba(X_input).max())
    advice = treatment_advice[status][lang]
    return jsonify({
        'status': status,
        'confidence': confidence,
        'advice': advice,
        'language': lang
    })
```

## 11. Summary

- Dataset: Plant Health Data from Kaggle
- Features used: Soil_Moisture, Ambient_Temperature, Humidity, Soil_pH
- Target: Plant_Health_Status (Healthy / Moderate Stress / High Stress)
- Best model: Random Forest
- Output: bilingual treatment advice (English + Arabic)
- Saved artifacts: `treatment_model.pkl`, `treatment_advice.json`, `treatment_features.json`

Drop the three files in the AI service `models/` folder and add the Flask endpoint code in section 10.